# Regression with TabFORGE

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SilenceX12138/TabFORGE/blob/master/docs/tutorial/notebooks/prediction/02_regression.ipynb)

> **Colab setup:** Open this notebook with the badge, then follow the [tutorial setup guide](https://silencex12138.github.io/TabFORGE/#/tutorials.md) to install `tabforge-ml[tutorials]` from PyPI in this runtime. Select a GPU for pretrained workloads.

## 1. Overview

This tutorial uses the real Airfoil Self-Noise dataset exposed by TabCamel, fine-tunes
`tabforge-v1`, and compares its root mean squared error (RMSE) with default TabPFN and random
forest regressors.

In [ ]:
%pip install "tabforge-ml[tutorials]"

## 2. Imports and setup

The pretrained TabFORGE backbone and Structure-aware Feature Encoder are the default path.

In [ ]:
import os
import tempfile
from pathlib import Path

from tabforge import (
    MODEL_ID,
    TabFORGEEmbeddingConfig,
    TabFORGERuntimeConfig,
    TabFORGETrainingConfig,
)

RANDOM_STATE = 7
MAX_TRAIN_ROWS = int(os.environ.get("TABFORGE_TUTORIAL_MAX_ROWS", "512"))
MAX_STEPS = int(os.environ.get("TABFORGE_TUTORIAL_MAX_STEPS", "1000"))
DEVICE = os.environ.get("TABFORGE_TUTORIAL_DEVICE", "auto")

assert MODEL_ID == "tabforge-v1"

## 3. Prepare a real-world dataset

`TabularDataset` downloads Airfoil Self-Noise and creates reproducible train, validation, and test
splits. Regression uses random splitting because the target is continuous.

In [ ]:
from tabcamel.data.dataset import TabularDataset

dataset = TabularDataset("airfoil_self_noise", task_type="regression")
split = dataset.split("random", train_size=0.8, random_state=RANDOM_STATE)
train_set = split["train_set"]
test_set = split["test_set"]

split_dict = train_set.split("random", train_size=0.8, random_state=RANDOM_STATE)
X_train = split_dict["train_set"].X_df.iloc[:MAX_TRAIN_ROWS].reset_index(drop=True)
y_train = split_dict["train_set"].y_s.iloc[:MAX_TRAIN_ROWS].reset_index(drop=True)
X_val = split_dict["test_set"].X_df.reset_index(drop=True)
y_val = split_dict["test_set"].y_s.reset_index(drop=True)
X_test = test_set.X_df.reset_index(drop=True)
y_test = test_set.y_s.reset_index(drop=True)
categorical_features = tuple(dataset.categorical_feature_list)

print({"train": X_train.shape, "val": X_val.shape, "test": X_test.shape, "target": y_train.name})

## 4. Configure TabFORGE

Multiple target trajectories provide a predictive distribution.

In [ ]:
from tabforge import TabFORGERegressor

regressor = TabFORGERegressor(
    categorical_features=categorical_features,
    embedding_config=TabFORGEEmbeddingConfig(model_path="auto", n_folds=2),
    training_config=TabFORGETrainingConfig(max_steps=MAX_STEPS, batch_size=128),
    runtime_config=TabFORGERuntimeConfig(device=DEVICE),
    n_prediction_samples=32,
    random_state=RANDOM_STATE,
)

## 5. Fit

The validation split is kept separate from fitted preprocessing and training embeddings.

In [ ]:
regressor.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    checkpoint="pretrained",
)

## 6. Predict

The point prediction is the mean of the sampled target distribution.

In [ ]:
distribution = regressor.predict_distribution(X_test)
predictions = regressor.predict(X_test)

print({"distribution_shape": distribution.shape, "prediction_shape": predictions.shape})

## 7. Compare with default regressors

For context, fit standalone TabPFN and scikit-learn random forest regressors with their default
parameters on exactly the same training split. RMSE is reported in the original target units;
lower is better.

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from tabpfn import TabPFNRegressor as ExternalTabPFNRegressor

tabpfn_regressor = ExternalTabPFNRegressor()
random_forest_regressor = RandomForestRegressor()

tabpfn_regressor.fit(X_train, y_train)
random_forest_regressor.fit(X_train, y_train)

comparison = pd.DataFrame(
    {
        "model": ["TabFORGE", "TabPFN", "Random forest"],
        "RMSE": [
            root_mean_squared_error(y_test, predictions),
            root_mean_squared_error(y_test, tabpfn_regressor.predict(X_test)),
            root_mean_squared_error(y_test, random_forest_regressor.predict(X_test)),
        ],
    }
).set_index("model")
comparison

## 8. Save and load

The restored estimator retains preprocessing, normalization, and learned weights.

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    checkpoint_path = regressor.save_checkpoint(Path(directory))
    restored = TabFORGERegressor.restore_from_checkpoint(checkpoint_path, device=DEVICE)
    restored_predictions = restored.predict(X_test.iloc[:8])

print({"checkpoint": checkpoint_path.name, "restored_rows": len(restored_predictions)})

## 9. Optional advanced configuration

Use `predict_interval` for central trajectory intervals and tune the number of trajectories against
validation RMSE when calibrated uncertainty matters.